# Text Feature Engineering Assignment
### Real-world Dataset: Amazon Product Reviews
**Tasks covered:** Preprocessing → Vocabulary → OHE / BoW / TF-IDF → Comparison → Sparse Matrix → Industry Q&A → Sentiment Classification

---
## 0. Setup – Install & Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install requests beautifulsoup4 pandas numpy scikit-learn nltk matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# NLP
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

print('All libraries loaded successfully.')

---
## 1. Dataset Collection

We simulate scraping by generating 120 realistic product reviews with labels.  
To scrape real data, replace the cell below with the scraping code provided in **Section 1b**.

In [ ]:
# ── 1a. Simulated dataset (120 reviews) ──────────────────────────────────────
positive_reviews = [
    "This product is absolutely fantastic! Works perfectly and arrived quickly.",
    "Great quality for the price. I'm very happy with this purchase.",
    "Excellent build quality. Would definitely recommend to friends.",
    "Super fast delivery and the product exceeded my expectations.",
    "Very satisfied with this item. It works exactly as described.",
    "Amazing product! The battery life is outstanding.",
    "Love this product. Easy to set up and use every day.",
    "Perfect gift! My family was thrilled with the quality.",
    "Outstanding performance. Best purchase I've made this year.",
    "Five stars! Packaging was great and the item arrived in perfect condition.",
    "Really impressed with the durability. Worth every penny.",
    "Smooth performance, great value, highly recommended.",
    "Beautifully designed product. Feels premium and works flawlessly.",
    "Exactly what I was looking for. Arrived ahead of schedule.",
    "Very comfortable and well-made. Will buy again.",
    "Works like a charm. Solid construction, no complaints.",
    "Brilliant product at a competitive price. Very impressed.",
    "Sturdy and reliable. Happy with the purchase overall.",
    "Incredible value for money. Setup was a breeze.",
    "Great product. Instructions were clear and it works perfectly.",
    "Loved it! Compact, lightweight, and very effective.",
    "Good quality materials and fast shipping. Very pleased.",
    "Top-notch product. Would order from this seller again.",
    "Absolutely worth the money. Performs better than expected.",
    "Really good product. Very easy to use and looks great.",
    "Highly recommended. Perfect for everyday use.",
    "Excellent product. Does exactly what it says on the tin.",
    "Quick delivery, well-packaged, and great performance.",
    "Solid product. Battery lasts long and screen is crisp.",
    "Very nice design and excellent functionality. Happy purchase.",
    "Loved the product. Customer support was also very helpful.",
    "Premium feel at an affordable price. Great buy.",
    "This works wonderfully. Far exceeded my expectations.",
    "Beautiful product. Easy to use and works as advertised.",
    "Super impressed with the quality. Will definitely reorder.",
    "Fast shipping and item is exactly as described. Happy!",
    "Wonderful experience from order to delivery. Loving the product.",
    "Great product for daily use. Comfortable and well-built.",
    "Solid build, smooth performance. Highly satisfied.",
    "One of the best purchases ever. Simply outstanding quality.",
    "Very good product. Does what it promises.",
    "Easy setup and works great. Totally worth the price.",
    "Exceptional quality. Better than I expected.",
    "Love the design and the functionality. Will recommend.",
    "Fantastic! All features work as described. Great value.",
    "Well-packaged and the product is of excellent quality.",
    "Very durable and efficient. A great buy for sure.",
    "Impressed by performance. Looks premium too.",
    "Really pleased with this. Fast delivery and great quality.",
    "Superb product. Works seamlessly. Totally worth it.",
    "Perfect for my needs. Quick delivery and good packaging.",
    "Very happy with this product. Great value for money.",
    "Excellent choice. Will be buying more from this brand.",
    "Reliable and efficient. No issues so far.",
    "Highly impressed. One of the best buys this season.",
    "Sturdy build, great performance. Completely satisfied.",
    "Works as expected. Good quality and great price.",
    "So happy with this product. Exceeded all my expectations.",
    "Fantastic quality! Everything is as described in listing.",
    "Delivered on time and works perfectly. Very satisfied.",
]

negative_reviews = [
    "Terrible product. Stopped working after just two days.",
    "Very disappointed. The quality is nothing like the pictures.",
    "Waste of money. Product broke within a week.",
    "Awful experience. Customer support was unhelpful and rude.",
    "Do not buy this. It's a complete scam.",
    "Poor build quality. Fell apart within days.",
    "Received a damaged product. Very unhappy with the purchase.",
    "Misleading description. Product is nothing like advertised.",
    "Cheap materials and bad design. Not worth the price at all.",
    "Very bad quality. Returned immediately.",
    "Stopped functioning after first use. Extremely frustrated.",
    "Would give zero stars if I could. Useless product.",
    "Took forever to deliver and arrived broken. Terrible.",
    "Complete disappointment. Not as described at all.",
    "Bad quality and terrible packaging. Will not buy again.",
    "Product malfunctioned on day one. Avoid this seller.",
    "Very flimsy and cheaply made. Regret this purchase.",
    "Worst purchase ever. Product doesn't work at all.",
    "Extremely poor quality. Not recommended.",
    "Broken out of the box. Had to return it immediately.",
    "Fraudulent product. Don't waste your money.",
    "Delivery was late and the item was defective.",
    "Not satisfied at all. Nothing worked as promised.",
    "Very unhappy. Product smelled bad and felt cheap.",
    "Useless item. Doesn't do what it claims to do.",
    "Pathetic quality. Broke within days of use.",
    "Horrible product. Returning it immediately.",
    "Complete waste of money. Avoid at all costs.",
    "Item arrived damaged and support didn't care at all.",
    "Terrible build quality. Feels like it will break any moment.",
    "Really bad product. Doesn't match the description.",
    "Disappointed with the quality. Way too expensive for what it is.",
    "Defective unit. Seller refused to replace it.",
    "Not worth it. Cheap construction and poor performance.",
    "Product failed immediately. Very frustrating experience.",
    "Awful product. Returning this right away.",
    "Fragile and poorly designed. Don't recommend this at all.",
    "Trash product. Worked for one hour and died.",
    "Shameful quality. Would never buy from this brand again.",
    "Arrived broken and seller is unresponsive. Disaster.",
    "Extremely disappointed. Product is a total joke.",
    "Malfunctioned immediately. Refund process was also frustrating.",
    "Not even close to what was advertised. Terrible.",
    "Feels very cheap and flimsy. Very dissatisfied.",
    "Product cracked on first use. Absolutely awful quality.",
    "False advertising. Product is a low-quality fake.",
    "Very bad experience overall. Nothing worked properly.",
    "Poor quality control. Half the features don't work.",
    "Item stopped working immediately. Waste of money.",
    "Deeply unsatisfied. Product is dangerous and defective.",
    "Substandard product. Packaging was also terrible.",
    "Cheap knockoff. Nothing like the original.",
    "Did not function at all. Support was non-existent.",
    "Completely broken. Money wasted.",
    "Quality is far below expectations. Not recommended.",
    "Terrible customer experience and terrible product.",
    "So bad it's embarrassing. Would not recommend to anyone.",
    "Broke on first use. Seller should be ashamed.",
    "Very poor quality. Arrived late and wasn't even working.",
    "Complete garbage. Avoid this product at all costs.",
]

all_reviews = positive_reviews + negative_reviews
all_labels  = ['positive'] * len(positive_reviews) + ['negative'] * len(negative_reviews)

df = pd.DataFrame({'review_text': all_reviews, 'sentiment': all_labels})
df.to_csv('reviews.csv', index=False)

print(f'Dataset shape : {df.shape}')
print(f'Sentiment dist:\n{df["sentiment"].value_counts()}')
df.head()

In [ ]:
# ── 1b. OPTIONAL: Real scraping with BeautifulSoup ───────────────────────────
# Uncomment and adapt this block to scrape real Amazon/Flipkart reviews.

# import requests
# from bs4 import BeautifulSoup
#
# HEADERS = {
#     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
#     "Accept-Language": "en-US,en;q=0.9"
# }
#
# def scrape_amazon_reviews(product_url: str, pages: int = 5) -> list[str]:
#     reviews = []
#     for page in range(1, pages + 1):
#         url = f"{product_url}&pageNumber={page}"
#         response = requests.get(url, headers=HEADERS, timeout=10)
#         soup = BeautifulSoup(response.content, 'html.parser')
#         for tag in soup.select('span[data-hook="review-body"] span'):
#             text = tag.get_text(strip=True)
#             if text:
#                 reviews.append(text)
#     return reviews
#
# PRODUCT_URL = 'https://www.amazon.in/.../product-reviews/ASIN/?reviewerType=all_reviews'
# raw_reviews = scrape_amazon_reviews(PRODUCT_URL, pages=10)
#
# df = pd.DataFrame({'review_text': raw_reviews})
# df.to_csv('reviews.csv', index=False)
# print(f'Scraped {len(df)} reviews')

---
## Task 1: Text Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
STOP_WORDS  = set(stopwords.words('english'))

def preprocess(text: str, remove_stopwords: bool = True, lemmatize: bool = True) -> str:
    """Full preprocessing pipeline."""
    # Step 1 – Lowercase
    text = text.lower()
    # Step 2 – Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Step 3 – Remove digits
    text = re.sub(r'\d+', '', text)
    # Step 4 – Tokenize
    tokens = word_tokenize(text)
    # Step 5 – (Optional) Remove stopwords
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOP_WORDS]
    # Step 6 – (Optional) Lemmatize
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df['cleaned_text'] = df['review_text'].apply(preprocess)

print('=== Before preprocessing ===')
print(df['review_text'].iloc[0])
print('\n=== After preprocessing ===')
print(df['cleaned_text'].iloc[0])

---
## Task 2: Vocabulary Creation

In [ ]:
from collections import Counter

all_tokens = ' '.join(df['cleaned_text']).split()
vocab_counter = Counter(all_tokens)
vocab = sorted(vocab_counter.keys())
word2idx = {w: i for i, w in enumerate(vocab)}

print(f'Vocabulary size : {len(vocab)}')
print(f'Top 20 frequent words:')
top20 = vocab_counter.most_common(20)
print(pd.DataFrame(top20, columns=['Word', 'Frequency']).to_string(index=False))

In [ ]:
# Bar chart of top 20 words
words, freqs = zip(*top20)
plt.figure(figsize=(12, 5))
plt.bar(words, freqs, color='steelblue', edgecolor='white')
plt.title('Top 20 Most Frequent Words', fontsize=14, fontweight='bold')
plt.xlabel('Word')
plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('top20_words.png', dpi=150)
plt.show()
print('Chart saved as top20_words.png')

---
## Task 3: Feature Engineering
### 3a – One Hot Encoding (OHE)

In [ ]:
def one_hot_encode(text: str, word2idx: dict) -> np.ndarray:
    """Returns a binary vector: 1 if word appears in document, 0 otherwise."""
    vec = np.zeros(len(word2idx), dtype=int)
    for word in text.split():
        if word in word2idx:
            vec[word2idx[word]] = 1
    return vec

ohe_matrix = np.array([one_hot_encode(text, word2idx) for text in df['cleaned_text']])

print(f'OHE Matrix Shape : {ohe_matrix.shape}')
print(f'OHE vector for Review #1 (non-zero positions) :')
nz = np.where(ohe_matrix[0] == 1)[0]
print({vocab[i]: 1 for i in nz[:10]}, '...')

### 3b – Bag of Words (BoW)

In [ ]:
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(df['cleaned_text'])

print(f'BoW Matrix Shape  : {bow_matrix.shape}')
print(f'Vocabulary size   : {len(bow_vectorizer.vocabulary_)}')
# Dense preview for first review
bow_df = pd.DataFrame(
    bow_matrix[0].toarray(),
    columns=bow_vectorizer.get_feature_names_out()
)
non_zero_bow = bow_df.loc[:, (bow_df != 0).any()]
print('\nNon-zero BoW features for Review #1:')
print(non_zero_bow)

### 3c – TF-IDF

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['cleaned_text'])

print(f'TF-IDF Matrix Shape : {tfidf_matrix.shape}')
tfidf_df = pd.DataFrame(
    tfidf_matrix[0].toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)
non_zero_tfidf = tfidf_df.loc[:, (tfidf_df != 0).any()].sort_values(by=0, axis=1, ascending=False)
print('\nTop TF-IDF features for Review #1:')
print(non_zero_tfidf)

---
## Task 4: Comparison Analysis

In [ ]:
comparison = pd.DataFrame({
    'Feature'       : ['Full Name', 'Value Type', 'Captures Frequency', 
                       'Word Importance', 'Handles Common Words',
                       'Dimensionality', 'Best Use Case'],
    'One Hot Encoding (OHE)': [
        'One Hot Encoding',
        'Binary (0 or 1)',
        'No – only presence',
        'No weighting',
        'No – all words treated equally',
        'Very high (full vocab)',
        'Categorical NLP, word embeddings base'
    ],
    'Bag of Words (BoW)': [
        'Bag of Words',
        'Integer (word count)',
        'Yes – raw counts',
        'No weighting',
        'No – common words get high counts',
        'High (full vocab)',
        'Document classification, spam filtering'
    ],
    'TF-IDF': [
        'Term Freq–Inverse Doc Freq',
        'Float (0.0–1.0)',
        'Yes – normalized',
        'Yes – rare words weighted higher',
        'Yes – common words penalized',
        'High (full vocab)',
        'Information retrieval, search engines'
    ]
})

print(comparison.to_string(index=False))

In [ ]:
# TF-IDF: which words are most important across all docs?
feature_names = tfidf_vectorizer.get_feature_names_out()
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_tfidf_df = pd.DataFrame({'Word': feature_names, 'Mean TF-IDF': mean_tfidf})
top_tfidf_df = top_tfidf_df.sort_values('Mean TF-IDF', ascending=False).head(15)

plt.figure(figsize=(12, 5))
plt.bar(top_tfidf_df['Word'], top_tfidf_df['Mean TF-IDF'], color='darkorange', edgecolor='white')
plt.title('Top 15 Words by Mean TF-IDF Score', fontsize=14, fontweight='bold')
plt.xlabel('Word')
plt.ylabel('Mean TF-IDF')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('top_tfidf_words.png', dpi=150)
plt.show()

print('''
Observation: Words with high TF-IDF scores appear frequently in only a few
documents. Common words (e.g., "product", "good") that appear in almost every
review receive a near-zero IDF weight because log(N/df) ≈ 0 when df ≈ N.
This penalization ensures that truly distinctive words drive the representation.
''')

---
## Task 5: Sparse Matrix Analysis

In [ ]:
def sparsity(matrix) -> float:
    """Percentage of zero elements."""
    total = matrix.shape[0] * matrix.shape[1]
    non_zero = matrix.nnz
    return (1 - non_zero / total) * 100

from scipy.sparse import issparse

# OHE (dense numpy array)
ohe_zeros   = np.sum(ohe_matrix == 0)
ohe_total   = ohe_matrix.size
ohe_sparsity = ohe_zeros / ohe_total * 100

print('='*50)
print(f'Matrix         Shape                 Sparsity')
print('='*50)
print(f'OHE            {ohe_matrix.shape}          {ohe_sparsity:.2f}%')
print(f'BoW            {bow_matrix.shape}       {sparsity(bow_matrix):.2f}%')
print(f'TF-IDF         {tfidf_matrix.shape}       {sparsity(tfidf_matrix):.2f}%')
print('='*50)

print('''
Why sparse matrices are inefficient for large-scale systems:
──────────────────────────────────────────────────────────
• In large corpora (millions of docs, hundreds of thousands of unique words)
  dense matrices would require enormous RAM: 1M docs × 500K vocab × 8 bytes
  = ~4 TB. Storing mostly zeros wastes memory.
• Arithmetic operations on zeros are redundant computations.
• Solution: use scipy.sparse (CSR/CSC format) which stores only non-zero values,
  or move to dense embeddings (Word2Vec, BERT) of fixed low dimension (e.g., 768).
''')

---
## Task 6: Real-world Questions

In [ ]:
print('''
Q1. Why does Bag of Words fail to capture semantic meaning?
────────────────────────────────────────────────────────────
BoW treats each word as an independent, unordered token. It ignores:
  • Context  : "The bank can guarantee deposits" vs "He sat on the river bank"
  • Synonymy : "happy" and "joyful" are unrelated in BoW space but mean the same.
  • Antonyms : "good" and "not good" have zero overlap but opposite meanings.
  • Word order: "dog bites man" = "man bites dog" in BoW.
Example: "This movie is not bad" and "This movie is bad" share almost all words
but have opposite meanings — BoW cannot distinguish them.

Q2. When to use BoW vs TF-IDF in industry?
────────────────────────────────────────────────────────────
  BoW   → Spam filtering, short text classification, simple sentiment tasks.
           Fast to compute. Works when relative frequency matters.
  TF-IDF→ Search engines (BM25 is TF-IDF variant), document ranking, keyword
           extraction, topic modelling. Better when you need to highlight distinctive
           terms across a large corpus.

Q3. Limitations of TF-IDF in real applications
────────────────────────────────────────────────────────────
  1. No semantic understanding ("king" and "queen" are unrelated vectors).
  2. Fails on short texts where IDF estimates are noisy.
  3. Vocabulary mismatch: unseen words at inference time get no vector.
  4. Cannot capture negation or sarcasm.
  5. High dimensionality → memory intensive at scale.
  6. Language-dependent; requires separate handling for multilingual corpora.
''')

---
## Task 7: Mini Use Case – Sentiment Classification

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(df['sentiment'])   # positive=1, negative=0

# ── Split ────────────────────────────────────────────────────────────────────
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['cleaned_text'], y, test_size=0.25, random_state=42, stratify=y
)

# ── Vectorise ────────────────────────────────────────────────────────────────
bow_vec   = CountVectorizer()
tfidf_vec = TfidfVectorizer()

X_train_bow   = bow_vec.fit_transform(X_train_raw)
X_test_bow    = bow_vec.transform(X_test_raw)

X_train_tfidf = tfidf_vec.fit_transform(X_train_raw)
X_test_tfidf  = tfidf_vec.transform(X_test_raw)

# ── Train & evaluate ─────────────────────────────────────────────────────────
results = []

for feat_name, X_tr, X_te in [
    ('BoW',   X_train_bow,   X_test_bow),
    ('TF-IDF', X_train_tfidf, X_test_tfidf)
]:
    for clf_name, clf in [
        ('Logistic Regression', LogisticRegression(max_iter=1000)),
        ('Naive Bayes',         MultinomialNB())
    ]:
        clf.fit(X_tr, y_train)
        preds = clf.predict(X_te)
        acc   = accuracy_score(y_test, preds)
        results.append({'Features': feat_name, 'Classifier': clf_name, 'Accuracy': round(acc*100, 2)})
        print(f'\n── {feat_name} + {clf_name} ──')
        print(classification_report(y_test, preds, target_names=le.classes_))

results_df = pd.DataFrame(results)
print('\n=== Summary Table ===')
print(results_df.to_string(index=False))

In [ ]:
# Accuracy comparison bar chart
pivot = results_df.pivot(index='Classifier', columns='Features', values='Accuracy')
ax = pivot.plot(kind='bar', figsize=(8, 5), colormap='Set2', edgecolor='white', rot=0)
ax.set_title('Sentiment Classification Accuracy: BoW vs TF-IDF', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 110)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)
plt.tight_layout()
plt.savefig('classification_accuracy.png', dpi=150)
plt.show()
print('Chart saved as classification_accuracy.png')

---
## Summary & Observations

| Feature Method | Pros | Cons |
|---|---|---|
| **OHE** | Simple, binary presence | Loses frequency; very high-dimensional |
| **BoW** | Captures counts; fast | No semantics; common words dominate |
| **TF-IDF** | Weights rare, informative words | Still no semantic understanding; high-dim |

**Key takeaways:**
1. TF-IDF generally outperforms BoW for sentiment tasks because it down-weights uninformative frequent words.
2. Logistic Regression benefits more from TF-IDF normalization; Naive Bayes can perform competitively with raw counts.
3. All three methods produce very sparse matrices—at scale, dense embeddings (Word2Vec, BERT) are preferred.
4. For production sentiment analysis, transformer-based models (e.g., RoBERTa) are the current best practice.